# 실습(빈칸) — 글자가 아니라 '의미'로 사내규정 찾기

**현대경제연구소 사내 헬프데스크 챗봇 '현경봇' 만들기 (3교시)**

> ⚠️ 이 노트북의 규정은 **교육용으로 만든 가상 규정**입니다.

## 시나리오

2교시에 규정을 청크로 쪼개고 키워드로 찾아봤습니다. 그런데 직원들은 규정집 말투로 묻지 않습니다.

| 김연구 씨가 쓰는 말 | 규정에 적힌 말 | 키워드 검색 |
|---------------------|----------------|-------------|
| 집에서 일해도 돼요? | 재택근무는 주 2회까지 | ❌ |
| 휴가 남은 거 내년에 써도 돼요? | 미사용 연차 최대 5일 이월 | ❌ |
| 법카로 술값 계산해도 돼요? | 법인카드 주류 단독 결제 불가 | ❌ |
| 챗GPT에 자료 넣어도 돼요? | 외부 생성형 AI에는 공개 자료만 | ❌ |

오늘은 **임베딩**과 **벡터 DB(Chroma)** 로 "뜻이 통하면 찾는" 검색을 만듭니다.
이번 시간에는 규정에 **연구 수행·발간 규정(`06_research.md`)** 이 추가되어 6개 문서를 다룹니다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | 키워드 검색의 한계, 임베딩과 벡터 DB |
| 2️⃣ 기본 | 문장을 숫자로 바꾸고 유사도 재보기 |
| 3️⃣ 응용 | 벡터 DB 저장·검색, k, filter, 유사도 점수 |
| 4️⃣ 실습 | 사내규정 6개를 의미로 검색하기 |
| 5️⃣ 미션 | 직원 말투 질문 10개로 검색 정확도 측정 |

## 📝 실습 방법

- 코드 속 `____` (밑줄 4개)를 알맞은 코드로 바꾼 뒤 실행하세요.
- 💡 막히면 **위니브 카페 노트북([Day5_3])** 의 같은 셀을 참고하세요.

# 1️⃣ 단계 — 이론

## 키워드 검색의 한계

2교시 셀 12에서 `"재택" in 청크`로 찾았습니다. 하지만 김연구 씨가 **"집에서 일해도 돼요?"** 라고 물으면
질문에 '재택'이라는 글자가 없어서 **답이 있는데도 못 찾습니다.** 컴퓨터에게 `집에서 일하기 ≠ 재택근무`입니다.

## 해결책 ① 임베딩 — 문장의 뜻을 좌표로

**임베딩(Embedding)** = 문장의 **뜻**을 숫자 묶음(좌표)으로 바꾸는 것.

```
"집에서 일해도 돼요?" ──●
                         ╲ 가까움 (뜻이 비슷)
"재택근무 신청"   ────────●
"해외 출장 숙박비" ───────────────● 멂 (뜻이 다름)
```

뜻이 비슷하면 좌표가 가깝고, 가까움은 **코사인 유사도**(1에 가까울수록 비슷)로 잽니다.

## 해결책 ② 벡터 DB — 가까운 청크를 빨리 찾는 서가

청크가 수천 개여도 질문과 가까운 것을 빠르게 꺼내 주는 저장소입니다. 오늘은 입문용 **Chroma**를 씁니다.

```
[준비] 규정 청크 → 임베딩 → 벡터 DB에 저장 (꼬리표도 함께)
[검색] 직원 질문 → 임베딩 → 가까운 청크 k개 꺼내기 → (4교시) AI에게 근거로 전달
```

# 2️⃣ 단계 — 문장을 숫자로 바꿔보기

### 셀 1 — 도구 설치

In [ ]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters langchain-huggingface python-dotenv sentence-transformers

### 셀 2 — 임베딩 모델 준비 (키 없어도 OK)

- OpenAI 키가 있으면 → `text-embedding-3-small` (한국어 성능이 더 좋음)
- 키가 없으면 → 무료 다국어 모델 자동 사용

In [6]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from dotenv import load_dotenv
    load_dotenv()

# LangSmith 키가 없을 때 401 경고 방지
os.environ["LANGCHAIN_TRACING_V2"] = "false"
os.environ["LANGSMITH_TRACING"] = "false"

if os.environ.get("OPENAI_API_KEY"):
    from langchain_openai import OpenAIEmbeddings
    # TODO: OpenAI 임베딩 모델 이름 (힌트: "text-embedding-3-small")
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    print("✅ OpenAI 임베딩 사용")
else:
    from langchain_huggingface import HuggingFaceEmbeddings
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        encode_kwargs={"normalize_embeddings": True},   # 길이를 1로 맞춰 점수를 0~1로
    )
    print("✅ 무료 임베딩 사용 (처음 한 번 다운로드, 1~2분)")

✅ OpenAI 임베딩 사용


### 셀 3 — 문장이 숫자가 되는지 보기

In [7]:
# TODO: 질문 문장 하나를 임베딩하는 메서드 (힌트: embed_로 시작)
vector = embeddings.embed_query("재택근무 규정 알려줘")

print("숫자가 몇 개?:", len(vector), "개")
print("앞 5개만 보기:", [round(x, 3) for x in vector[:5]])

숫자가 몇 개?: 1536 개
앞 5개만 보기: [0.002, 0.01, 0.062, 0.011, 0.015]


> 💡 OpenAI 모델은 1536개, 무료 모델은 384개 숫자가 나옵니다. 이 숫자 묶음이 문장의 **의미 좌표**입니다.

### 셀 4 — '가까움'을 숫자로 재보기 (코사인 유사도)

In [9]:
import numpy as np

def similarity(text1, text2):
    v1 = np.array(embeddings.embed_query(text1))
    v2 = np.array(embeddings.embed_query(text2))
    # TODO: 코사인 유사도 = 내적 / (길이1 × 길이2)
    return float(v1 @ v2 / (np.linalg.norm(v1) * np.linalg.norm(v2)))

base = "집에서 일해도 돼요?"
# TODO: 비교할 문장을 하나 더 추가해 보세요
for other in ["재택근무 신청 방법", "해외 출장 숙박비 한도", "직장 내 출퇴근 시간"]:
    print(f"'{base}' ↔ '{other}': {similarity(base, other):.3f}")

'집에서 일해도 돼요?' ↔ '재택근무 신청 방법': 0.199
'집에서 일해도 돼요?' ↔ '해외 출장 숙박비 한도': 0.280
'집에서 일해도 돼요?' ↔ '직장 내 출퇴근 시간': 0.241


> 💡 글자는 하나도 안 겹치는데 '재택근무 신청 방법'의 점수가 가장 높습니다. 이게 의미 검색이 되는 이유입니다.

# 3️⃣ 단계 — 벡터 DB에 저장하고 검색하기

### 셀 5 — 연습용 규정 문장을 벡터 DB에 저장

In [10]:
from langchain_core.documents import Document
from langchain_chroma import Chroma

sample_docs = [
    Document(page_content="재택근무는 주 2회까지 가능하며 전날 17시까지 신청합니다.", metadata={"topic": "leave"}),
    Document(page_content="택시는 대중교통이 없거나 22시 이후에만 출장비로 인정됩니다.", metadata={"topic": "travel"}),
    # TODO: security 분류 문장 하나를 만들어 보세요
    Document(page_content="보안 규정에 따라 모든 직원은 로그인 시 강력한 비밀번호를 사용해야 합니다.", metadata={"topic": "security"}),
]

# TODO: Document 리스트로 벡터 DB를 만드는 메서드 (힌트: from_으로 시작)
db = Chroma.from_documents(
    sample_docs, embeddings,
    collection_name="hri_sample",
    collection_metadata={"hnsw:space": "cosine"},   # 코사인 유사도로 거리 계산
    ids=["s1", "s2", "s3"],
)
print("벡터 DB에 문장 3개 저장 완료 ✅")

벡터 DB에 문장 3개 저장 완료 ✅


> 💡 `ids`를 지정하지 않으면 셀을 다시 실행할 때마다 같은 문장이 또 저장돼서 검색 결과가 중복됩니다.

### 셀 6 — 의미로 검색 (글자가 달라도 찾는다)

"밤늦게 택시 타면 회사 돈으로 되나요?" — 질문에 '출장', '22시', '대중교통'이라는 글자가 없습니다.

In [11]:
query = "밤늦게 택시 타면 회사 돈으로 되나요?"

# TODO: 의미로 가장 가까운 문장 1개 찾기
result = db.similarity_search(query, k=1)

print("질문:", query)
print("찾은 문장:", result[0].page_content)
print("분류:", result[0].metadata["topic"])

질문: 밤늦게 택시 타면 회사 돈으로 되나요?
찾은 문장: 택시는 대중교통이 없거나 22시 이후에만 출장비로 인정됩니다.
분류: travel


> 💡 '밤늦게'와 '22시 이후', '회사 돈'과 '출장비'를 뜻으로 연결해서 **travel 문장**을 찾았습니다.

### 셀 7 — 검색 개수(k) 바꿔보기

In [12]:
query = "재택근무 규정 알려줘"

# TODO: k를 1, 2, 3으로 바꿔가며 검색
for k in [1, 2, 3]:
    results = db.similarity_search(query, k=k)
    print(f"k={k} → 가져온 분류: {[r.metadata['topic'] for r in results]}")

k=1 → 가져온 분류: ['leave']
k=2 → 가져온 분류: ['leave', 'security']
k=3 → 가져온 분류: ['leave', 'security', 'travel']


> 💡 k가 커지면 관련 없는 분류(travel, security)도 섞입니다. 섞인 청크를 AI에게 주면 엉뚱한 근거로 답할 수 있습니다.

### 셀 8 — 분류로 범위 좁히기 (filter)

In [13]:
query = "관련 규정 알려줘"

print("───[ 전체에서 검색 ]───")
for r in db.similarity_search(query, k=2):
    print(" -", r.metadata["topic"], ":", r.page_content)

print("\n───[ security 분류만 검색 ]───")
# TODO: topic이 security인 문장만 검색하도록 filter 지정
for r in db.similarity_search(query, k=2, filter={"topic": "security"}):
    print(" -", r.metadata["topic"], ":", r.page_content)

───[ 전체에서 검색 ]───
 - security : 보안 규정에 따라 모든 직원은 로그인 시 강력한 비밀번호를 사용해야 합니다.
 - leave : 재택근무는 주 2회까지 가능하며 전날 17시까지 신청합니다.

───[ security 분류만 검색 ]───
 - security : 보안 규정에 따라 모든 직원은 로그인 시 강력한 비밀번호를 사용해야 합니다.


### 셀 9 — 유사도 점수 보기 (관련도)

검색 결과가 **얼마나 관련 있는지** 점수로 봅니다. `similarity_search_with_relevance_scores`는 **0~1, 높을수록 관련**이 높습니다.
이 점수가 4교시에서 "관련 없는 청크는 근거로 넣지 않기"의 기준이 됩니다.

In [14]:
for query in ["집에서 근무하고 싶어요", "사내 헬스장 운영시간"]:
    print(f"질문: {query}")
    # TODO: (문서, 관련도 점수) 쌍을 돌려주는 검색 메서드
    for doc, score in db.similarity_search_with_score(query, k=3):
        print(f"   {score:.3f}  [{doc.metadata['topic']}] {doc.page_content[:30]}...")
    print()

질문: 집에서 근무하고 싶어요
   0.650  [leave] 재택근무는 주 2회까지 가능하며 전날 17시까지 신청합...
   0.790  [security] 보안 규정에 따라 모든 직원은 로그인 시 강력한 비밀번...
   0.931  [travel] 택시는 대중교통이 없거나 22시 이후에만 출장비로 인정...

질문: 사내 헬스장 운영시간
   0.692  [leave] 재택근무는 주 2회까지 가능하며 전날 17시까지 신청합...
   0.715  [travel] 택시는 대중교통이 없거나 22시 이후에만 출장비로 인정...
   0.835  [security] 보안 규정에 따라 모든 직원은 로그인 시 강력한 비밀번...



> 💡 규정에 없는 '헬스장' 질문은 1등 점수도 낮습니다. 하지만 `similarity_search`는 **점수가 낮아도 무조건 k개를 돌려줍니다.**
> 그래서 실무에서는 "점수가 기준보다 낮으면 버린다"는 **임계값**을 함께 씁니다.
>
> ⚠️ `similarity_search_with_score`는 **거리**(낮을수록 비슷)를 돌려주니 헷갈리지 마세요.

# 4️⃣ 단계 — 사내규정 6개를 의미로 검색하기

### 셀 10 — 규정 문서 준비 (2교시 내용 압축)

불러오기 → 쪼개기 → 꼬리표(분류·담당부서)를 한 셀에 모았습니다.

In [15]:
documents = {
    "01_faq.md": """# 현대경제연구소 사내 헬프데스크 자주 묻는 질문 (FAQ)

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

사내 헬프데스크 '현경봇'이 자주 받는 질문을 모았습니다.
자세한 내용은 각 규정 문서를 확인해 주세요.

---

### Q. 입사 첫날에는 무엇을 해야 하나요?
A. 오전 9시까지 본관 3층 인사팀으로 오시면 사원증, 노트북, 보안 USB를 지급합니다.
오후에는 정보보안 교육(2시간)을 반드시 이수해야 사내 시스템 계정이 활성화됩니다.

### Q. 그룹웨어 계정은 언제 생기나요?
A. 정보보안 교육 이수 후 당일 18시까지 생성됩니다. 초기 비밀번호는 문자로 안내되며,
최초 로그인 시 반드시 변경해야 합니다.

### Q. 연차·재택근무 신청은 어디서 하나요?
A. 그룹웨어 > 근태관리 메뉴에서 신청합니다. 신청 기한은 근무·휴가 규정을 확인해 주세요.

### Q. 출장비 정산은 어떻게 하나요?
A. 그룹웨어 > 경비정산 메뉴에서 영수증 사진을 첨부해 제출합니다.
출장 복귀 후 7영업일 이내에 정산해야 합니다.

### Q. 노트북이 고장 났어요.
A. IT헬프데스크(내선 1234)로 연락하시면 대체 노트북을 당일 지급합니다.
개인 PC로 업무 자료를 옮기는 것은 금지되어 있습니다.

### Q. 사내 도서관은 어떻게 이용하나요?
A. 본관 5층 자료실은 평일 09:00~20:00 운영합니다. 연구 자료는 1인 10권까지
2주간 대출할 수 있으며, 대외비 보고서는 열람만 가능합니다.

### Q. 외부 기고나 강연 요청이 왔어요.
A. 외부 활동은 사전에 본부장 승인과 홍보팀 협의가 필요합니다.
미발표 연구 결과나 전망 수치는 언급할 수 없습니다.

### Q. 명함은 어떻게 신청하나요?
A. 그룹웨어 > 총무 > 명함신청 메뉴에서 신청하면 5영업일 이내 자리로 배송됩니다.

### Q. 회의실 예약은 어떻게 하나요?
A. 그룹웨어 > 시설예약에서 예약합니다. 1회 최대 3시간까지 예약할 수 있으며,
대회의실(30인)은 팀장 이상만 예약할 수 있습니다.

### Q. 각 부서 연락처를 알려주세요.
A. 인사팀(내선 2100), 재무팀(내선 2200), IT헬프데스크(내선 1234),
총무팀(내선 2300), 홍보팀(내선 2400)입니다.
""",

    "02_leave.md": """# 현대경제연구소 근무·휴가 규정

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

본 규정은 현대경제연구소 전 직원(계약직 포함)에게 적용됩니다. 담당: 인사팀

## 제1조 근무시간

- 기본 근무시간은 09:00~18:00 이며, 점심시간은 12:00~13:00 입니다.
- **시차출퇴근제**: 08:00~10:00 사이에서 출근 시각을 선택하고, 하루 8시간을 근무합니다.
- 출근 시각은 월 단위로 정하며, 변경하려면 전월 25일까지 그룹웨어에 신청합니다.
- 연장근무는 사전에 팀장 승인을 받아야 하며, 주 12시간을 넘을 수 없습니다.

## 제2조 재택근무

- 재택근무는 **주 2회까지** 가능합니다.
- 전날 17시까지 그룹웨어에서 신청하고 팀장 승인을 받아야 합니다.
- **입사 3개월 미만 직원은 재택근무를 할 수 없습니다.**
- 재택근무 중에도 09:00~16:00(코어타임)에는 메신저 응답이 가능해야 합니다.
- 대외비 자료는 재택근무 중 열람할 수 없습니다. (정보보안 규정 참고)

## 제3조 연차휴가

| 근속 기간 | 연차 일수 |
|-----------|-----------|
| 1년 미만 | 1개월 개근 시 1일 (최대 11일) |
| 1년 이상 3년 미만 | 15일 |
| 3년 이상 5년 미만 | 16일 |
| 5년 이상 7년 미만 | 17일 |
| 7년 이상 | 2년마다 1일 가산 (최대 25일) |

- 반차(0.5일)는 오전(09:00~13:00), 오후(14:00~18:00) 단위로 사용합니다.
- **반반차(2시간 단위)는 없습니다.**
- 연차는 사용 **3영업일 전**까지 그룹웨어로 신청합니다.
- **3일 이상 연속** 사용 시 **2주 전**까지 신청해야 합니다.
- 긴급한 경우 당일 09시 전 팀장에게 연락하고, 복귀 후 2영업일 이내 사후 신청합니다.

## 제4조 미사용 연차

- 미사용 연차는 다음 해로 **최대 5일까지 이월**할 수 있습니다.
- 이월된 연차는 **다음 해 3월 31일까지** 사용해야 하며, 기한이 지나면 소멸합니다.
- 연차사용촉진제를 시행하므로, 이월분을 제외한 미사용 연차는 금전 보상하지 않습니다.
- 퇴직 시 남은 연차는 퇴직 정산 시 수당으로 지급합니다.

## 제5조 경조휴가 (유급)

| 구분 | 휴가 | 경조금 |
|------|------|--------|
| 본인 결혼 | 5일 | 50만원 |
| 자녀 결혼 | 1일 | 30만원 |
| 배우자 출산 | 10일 | 20만원 |
| 부모·배우자 부모 사망 | 5일 | 50만원 |
| 조부모·배우자 조부모 사망 | 3일 | 20만원 |
| 형제자매 사망 | 2일 | 10만원 |

- 경조휴가는 주말·공휴일을 제외하고 계산합니다.
- 배우자 출산휴가는 출산일로부터 90일 이내에 사용해야 하며, 1회 분할 사용이 가능합니다.
- 경조금은 증빙서류(청첩장, 가족관계증명서 등) 제출 후 다음 달 급여와 함께 지급합니다.

## 제6조 병가

- 유급 병가는 연 5일입니다. 이를 초과하면 연차에서 차감합니다.
- 3일 이상 연속 병가 시 진단서를 제출해야 합니다.
- 업무상 부상·질병은 산재 절차에 따르며, 인사팀에 즉시 알려야 합니다.

## 제7조 육아 지원

- 육아휴직: 만 8세 이하 자녀 1명당 최대 1년, 분할 사용 2회까지 가능합니다.
- 육아기 근로시간 단축: 하루 1~5시간 단축 근무를 신청할 수 있습니다.
- 신청은 시작일 30일 전까지 인사팀에 서면으로 합니다.
""",

    "03_travel.md": """# 현대경제연구소 출장·경비 규정

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

본 규정은 업무상 출장과 법인카드 사용에 적용됩니다. 담당: 재무팀

## 제1조 출장 신청

- 국내 출장: 출발 **3영업일 전**까지 그룹웨어 품의, 팀장 승인
- 해외 출장: 출발 **2주 전**까지 품의, 본부장 승인
- 해외 출장 품의서에는 일정표, 방문기관, 예상 경비를 반드시 적습니다.
- 출장 일정이 바뀌면 즉시 변경 품의를 올립니다.

## 제2조 국내 출장비

| 항목 | 기준 |
|------|------|
| 일비 | 1일 20,000원 |
| 식비 | 실비, 1끼 12,000원 한도 |
| 숙박비(서울) | 1박 120,000원 한도 |
| 숙박비(광역시) | 1박 100,000원 한도 |
| 숙박비(기타 지역) | 1박 80,000원 한도 |

- 당일 출장은 숙박비를 지급하지 않으며, 일비는 4시간 이상 출장 시 지급합니다.
- 숙박비 한도를 넘는 금액은 본인 부담입니다.
- 친척·지인 집에 묵는 경우 1박 30,000원을 정액 지급합니다.

## 제3조 교통비

- 철도: KTX·SRT 일반실 기준. 특실 이용 시 차액은 본인 부담입니다.
- 항공(국내): 제주 및 편도 4시간 이상 거리만 인정, 일반석 기준
- 자가용: km당 250원과 통행료를 지급합니다. 주차비는 실비로 인정합니다.
- 택시: 대중교통이 없는 구간이거나 22시 이후에만 인정됩니다.

## 제4조 해외 출장비

| 항목 | 북미·유럽 | 아시아 | 기타 지역 |
|------|-----------|--------|-----------|
| 일비 | USD 60 | USD 50 | USD 50 |
| 숙박비(1박 한도) | USD 200 | USD 150 | USD 150 |

- 항공권: 이코노미석 (임원은 비행 8시간 이상 시 비즈니스석)
- 항공권은 재무팀 지정 여행사를 통해 예약해야 합니다.
- 해외 여행자보험은 연구소가 일괄 가입합니다.
- 해외에서 법인카드 사용 시 환율은 카드사 청구 환율을 적용합니다.

## 제5조 경비 정산

- 출장 복귀 후 **7영업일 이내**에 영수증을 첨부해 정산합니다.
- 기한을 넘기면 재무팀 승인 후에만 정산이 가능합니다.
- 영수증을 분실한 경우 '지출증빙 확인서'를 작성하며, 건당 30,000원까지만 인정됩니다.
- 개인카드로 결제한 경비는 정산 승인 후 다음 급여일에 지급합니다.

## 제6조 법인카드 사용 제한

다음은 법인카드로 결제할 수 없습니다.
- 개인 용품, 상품권, 유흥업소
- 23시 이후 결제 (출장 중 불가피한 교통비 제외)
- 주류 단독 결제 (공식 회식 제외)
- 주말·공휴일 결제 (출장·행사 등 사전 승인 건 제외)

## 제7조 회식·접대비

- 팀 회식비: 분기별 1인당 30,000원
- 외부 접대비: 1인당 50,000원 한도, 초과 시 사전 본부장 승인 필요
- 접대비 정산 시 참석자 명단과 소속 기관을 반드시 적습니다.
- 공무원 등 청탁금지법 적용 대상자 접대 시 1인당 30,000원을 넘을 수 없습니다.
""",

    "04_security.md": """# 현대경제연구소 정보보안 규정

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

연구소의 연구 성과와 고객 정보를 보호하기 위한 규정입니다. 담당: IT팀(정보보안파트)

## 제1조 문서 보안등급

| 등급 | 설명 | 외부 반출 |
|------|------|-----------|
| 대외비 | 미발표 연구결과, 경제전망 수치, 고객사 자료 | 금지 |
| 사내한 | 내부 회의자료, 업무 매뉴얼 | 팀장 승인 시 가능 |
| 공개 | 발간된 보고서, 보도자료 | 가능 |

- 문서 작성자는 작성 시 반드시 보안등급을 표시해야 합니다.
- 대외비 문서 출력 시 워터마크(출력자 사번·일시)가 자동으로 찍힙니다.
- 대외비 문서는 사내망에서만 열람할 수 있으며, 재택근무 중에는 열람할 수 없습니다.

## 제2조 엠바고(보도 유예)

- 경제전망 수치는 **공식 발표 전 외부에 언급할 수 없습니다.**
- 가족·지인과의 대화, 개인 SNS도 외부 언급에 해당합니다.
- 언론 문의는 개인이 답하지 말고 반드시 **홍보팀(내선 2400)** 으로 연결합니다.
- 엠바고 위반은 징계 사유입니다.

## 제3조 외부 생성형 AI 사용

- ChatGPT 등 외부 AI에는 **공개 등급 자료만** 입력할 수 있습니다.
- 대외비 자료, 미발표 전망치, 고객사 정보, 개인정보 입력은 **금지**입니다.
- 대외비·사내한 자료는 사내 AI 서비스 **'현경 어시스턴트'** 로만 처리합니다.
- AI가 만든 결과물을 보고서에 쓸 때는 반드시 사실 확인을 거치고, 출처를 밝힙니다.

## 제4조 저장매체·기기

- 개인 USB 사용은 금지되며, IT팀이 지급한 보안 USB만 사용할 수 있습니다.
- 보안 USB는 사내 PC에서만 읽히며, 반출 시 자동 암호화됩니다.
- 업무용 노트북을 외부로 가져갈 때는 그룹웨어에서 사전에 반출 신청을 합니다.
- 해외 출장 시 노트북 반출은 팀장 승인 외에 정보보안파트 승인이 추가로 필요합니다.
- 노트북·보안 USB 분실 시 **24시간 이내** IT헬프데스크(내선 1234)에 신고합니다.

## 제5조 계정·비밀번호

- 비밀번호는 10자 이상, 영문·숫자·특수문자 조합이어야 합니다.
- 90일마다 변경해야 하며, 최근 3회 사용한 비밀번호는 다시 쓸 수 없습니다.
- 5회 연속 틀리면 계정이 잠기며, IT헬프데스크(내선 1234)로 해제를 요청합니다.
- 계정은 본인만 사용해야 하며, 동료와 공유하면 안 됩니다.

## 제6조 이메일·메신저

- 외부로 첨부파일을 보낼 때는 자동으로 팀장 결재가 요청됩니다.
- 출처가 불분명한 메일의 링크나 첨부파일은 열지 말고 IT헬프데스크에 신고합니다.
- 개인 이메일·개인 메신저로 업무 파일을 보내는 것은 금지입니다.

## 제7조 클린데스크

- 퇴근 시 책상 위에 서류를 두지 않고, 대외비 문서는 잠금 서랍에 보관합니다.
- 자리를 비울 때는 PC 화면을 잠급니다. (Windows 키 + L)
- 정보보안파트는 월 1회 불시 점검을 합니다.
""",

    "05_welfare.md": """# 현대경제연구소 복리후생 안내

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

직원의 건강과 성장을 지원하는 제도입니다. 담당: 인사팀(복리후생), 총무팀(시설)

## 1. 자기계발비

- 연 120만원 한도로 지원합니다. (입사 첫해는 입사 월부터 월할 계산)
- 대상: 어학, 통계·데이터 분석, 경제·금융 관련 강좌 및 도서
- 연구 관련 학회 참가비는 자기계발비와 별도로 업무 경비로 처리합니다.
- 영수증을 첨부해 그룹웨어 > 복리후생 메뉴에서 신청하며, 분기별로 지급합니다.
- 미사용 금액은 다음 해로 이월되지 않습니다.

## 2. 학위 취득 지원

- 근속 2년 이상 연구직은 석·박사 과정 학비의 50%를 지원받을 수 있습니다.
- 지원을 받은 경우 학위 취득 후 2년간 의무 근무해야 하며,
  그 전에 퇴직하면 지원금을 반환해야 합니다.

## 3. 건강검진

- 전 직원 연 1회 종합건강검진을 지원합니다. (지정 병원 3곳 중 선택)
- 만 40세 이상은 위·대장 내시경, 뇌 MRI 중 1개 항목을 추가로 선택할 수 있습니다.
- 배우자는 2년에 1회 기본 검진을 지원합니다.
- 검진 당일은 공가(유급)로 처리합니다.

## 4. 사내식당·카페

- 사내식당(본관 지하 1층): 조식 08:00~09:00, 중식 11:30~13:00
- 식대는 1끼 3,000원이며, 급여에서 공제됩니다. 조식은 무료입니다.
- 사내 카페(1층): 08:00~18:00, 직원 50% 할인

## 5. 주차 지원

- 사내 주차장은 임원, 임산부, 장애인 직원에게 우선 배정합니다.
- 일반 직원은 추첨으로 분기별 배정하며, 월 50,000원을 급여에서 공제합니다.
- 주차 신청은 분기 시작 2주 전까지 총무팀(내선 2300)에 합니다.
- 배정받지 못한 직원은 대중교통 이용을 권장하며, 별도 교통비는 지급하지 않습니다.

## 6. 휴양시설

- 제휴 콘도(강원·제주·경주)를 연 2회, 1회 최대 2박까지 이용할 수 있습니다.
- 성수기(7~8월, 12~2월)에는 추첨으로 배정합니다.
- 숙박비의 70%를 연구소가 지원합니다.

## 7. 경조 지원

- 경조휴가와 경조금은 근무·휴가 규정 제5조를 따릅니다.
- 직원 본인 사망 시 유가족에게 위로금 1,000만원을 지급합니다.
- 화환·조화는 인사팀이 대표로 보냅니다.
""",

    "06_research.md": """# 현대경제연구소 연구 수행·발간 규정

※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합니다.

연구 과제의 수행, 보고서 발간, 연구윤리에 관한 규정입니다. 담당: 연구기획팀

## 제1조 연구 과제 등록

- 모든 연구 과제는 착수 전 연구관리시스템(RMS)에 등록해야 합니다.
- 등록 항목: 과제명, 연구책임자, 기간, 예산, 보안등급, 외부 협력기관
- 외부 수탁 과제는 계약 체결 후 5영업일 이내에 등록합니다.
- 과제 기간이 1개월 이상 바뀌면 변경 신청을 합니다.

## 제2조 보고서 발간 절차

1. 초안 작성 (연구책임자)
2. 팀 내부 검토 (팀장)
3. **외부 전문가 감수** (정기 보고서는 1인 이상, 경제전망 보고서는 2인 이상)
4. 연구기획팀 형식 검토 (오탈자, 인용 형식, 그래프 출처)
5. 원장 최종 승인 후 발간

- 정기 보고서는 발간 예정일 **3주 전**까지 감수 단계에 들어가야 합니다.
- 경제전망 보고서는 발간 당일 09:00 이전에 외부에 공개할 수 없습니다. (정보보안 규정 엠바고 참고)

## 제3조 연구윤리

- 다른 사람의 글·데이터를 쓸 때는 반드시 출처를 밝힙니다.
- 발간 전 표절 검사 시스템으로 검사하며, **유사도 15% 이상**이면 수정 후 재검사합니다.
- 생성형 AI로 작성한 문장은 그대로 쓰지 않고, 사용 사실을 보고서 말미에 밝힙니다.
- 데이터 위·변조, 중복 게재는 징계 사유입니다.

## 제4조 외부 데이터 구입

- 유료 데이터베이스·통계 자료 구입은 연구기획팀을 통해 일괄 구매합니다.
- 1건 300만원 이상은 연구기획팀장 승인, 1,000만원 이상은 원장 승인이 필요합니다.
- 구입한 데이터는 라이선스 범위 안에서만 사용하며, 외부 공유를 금지합니다.

## 제5조 연구 성과 평가

- 연구 성과는 연 1회(12월) 평가합니다.
- 평가 항목: 보고서 발간 실적(40%), 언론 인용(20%), 외부 강연·기고(20%), 내부 기여(20%)
- 평가 결과는 다음 해 성과급과 승진 심사에 반영합니다.

## 제6조 외부 활동

- 외부 강연·기고·자문은 사전에 본부장 승인과 홍보팀 협의가 필요합니다.
- 외부 강연료는 1회 100만원까지 개인 수령할 수 있으며, 초과분은 연구소에 귀속됩니다.
- 언론 인터뷰는 반드시 홍보팀을 통해 진행합니다.
""",

}

import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

os.makedirs("hri_docs", exist_ok=True)
for filename, content in documents.items():
    with open(f"hri_docs/{filename}", "w", encoding="utf-8") as f:
        f.write(content)

# 불러오기 → 쪼개기
all_docs = DirectoryLoader("hri_docs", glob="*.md", loader_cls=TextLoader,
                           loader_kwargs={"encoding": "utf-8"}).load()
chunks = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30).split_documents(all_docs)

# 꼬리표 달기
category_map = {"01_faq.md": "faq", "02_leave.md": "leave", "03_travel.md": "travel",
                "04_security.md": "security", "05_welfare.md": "welfare", "06_research.md": "research"}
department_map = {"faq": "인사팀", "leave": "인사팀", "travel": "재무팀",
                  "security": "IT팀", "welfare": "인사팀·총무팀", "research": "연구기획팀"}
for c in chunks:
    c.metadata["category"] = category_map[os.path.basename(c.metadata["source"])]
    c.metadata["department"] = department_map[c.metadata["category"]]

print("불러온 문서:", len(all_docs), "개")
print("준비된 청크:", len(chunks), "개")

불러온 문서: 6 개
준비된 청크: 39 개


C:\Users\user\AppData\Local\Temp\ipykernel_30928\185215671.py:357: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


### 셀 11 — 규정 청크를 벡터 DB에 저장

(무료 모델은 1~2분 걸릴 수 있어요)

In [16]:
hri_db = Chroma.from_documents(
    chunks, embeddings,                       # TODO: 셀 10에서 만든 청크 리스트
    collection_name="hri_regulations",
    collection_metadata={"hnsw:space": "cosine"},
    ids=[f"chunk-{i}" for i in range(len(chunks))],
)
print("사내규정 청크 저장 완료 ✅")
print("저장된 청크 수:", hri_db._collection.count())

사내규정 청크 저장 완료 ✅
저장된 청크 수: 39


### 셀 12 — 키워드로 못 찾던 질문 (핵심!)

"휴가 남은 거 내년에 써도 돼요?" — 규정에는 '미사용 연차', '이월'이라고 적혀 있습니다.

In [18]:
query = "휴가 남은 거 내년에 써도 돼요?"

# TODO: hri_db에서 가까운 청크 3개 찾기
results = hri_db.similarity_search(query, k=3)

print("질문:", query, "\n")
for i, r in enumerate(results, 1):
    # TODO: 꼬리표에서 분류와 담당부서 꺼내기
    print(f"[{i}] (분류: {r.metadata['category']} / 문의: {r.metadata['department']})")
    print("   ", r.page_content[:80].replace("\n", " "), "...")

질문: 휴가 남은 거 내년에 써도 돼요? 

[1] (분류: leave / 문의: 인사팀)
    ## 제3조 연차휴가  | 근속 기간 | 연차 일수 | |-----------|-----------| | 1년 미만 | 1개월 개근 시 1일 ( ...
[2] (분류: leave / 문의: 인사팀)
    ## 제5조 경조휴가 (유급)  | 구분 | 휴가 | 경조금 | |------|------|--------| | 본인 결혼 | 5일 | 50만원 ...
[3] (분류: travel / 문의: 재무팀)
    ## 제2조 국내 출장비  | 항목 | 기준 | |------|------| | 일비 | 1일 20,000원 | | 식비 | 실비, 1끼 12, ...


> 💡 '이월'이라는 글자가 질문에 없는데도 **근무·휴가 규정**의 연차 청크를 찾고, 담당부서(인사팀) 꼬리표도 따라옵니다.

### 셀 13 — 키워드 검색 vs 의미 검색 (직접 비교)

질문: **"집에서 일해도 돼요?"**

In [19]:
q = "집에서 일해도 돼요?"

# (1) 키워드 검색: 질문의 단어가 하나라도 들어간 청크
words = ["집에서", "일해도"]
keyword_hits = [c for c in chunks if any(w in c.page_content for w in words)]
print("───[ 키워드 검색 ]───")
print(f"{words} 가 들어간 청크: {len(keyword_hits)}개 → 재택근무 조항 연결 실패 ❌")

# (2) 의미 검색
print("\n───[ 의미 검색 ]───")
# TODO: q로 의미 검색, 2개
for r in hri_db.similarity_search(q, k=2):
    print(f" - [{r.metadata['category']}]", r.page_content[:60].replace("\n", " "), "...")

───[ 키워드 검색 ]───
['집에서', '일해도'] 가 들어간 청크: 0개 → 재택근무 조항 연결 실패 ❌

───[ 의미 검색 ]───
 - [welfare] ## 4. 사내식당·카페  - 사내식당(본관 지하 1층): 조식 08:00~09:00, 중식 11:30~13 ...
 - [welfare] ## 5. 주차 지원  - 사내 주차장은 임원, 임산부, 장애인 직원에게 우선 배정합니다. - 일반 직원은  ...


### 셀 14 — 분류 필터로 정확도 높이기

"외부로 파일 보내도 돼요?"는 보안 질문입니다. 전체 검색과 `security` 필터 검색을 비교합니다.

In [20]:
q = "외부로 파일 보내도 돼요?"

print("───[ 전체 검색 ]───")
for r in hri_db.similarity_search(q, k=3):
    print(f" - [{r.metadata['category']}]", r.page_content[:50].replace("\n", " "), "...")

print("\n───[ security만 검색 ]───")
# TODO: category가 security인 청크만 검색
for r in hri_db.similarity_search(q, k=3, filter={"category": "security"}):
    print(f" - [{r.metadata['category']}]", r.page_content[:50].replace("\n", " "), "...")

───[ 전체 검색 ]───
 - [security] - 문서 작성자는 작성 시 반드시 보안등급을 표시해야 합니다. - 대외비 문서 출력 시 워 ...
 - [security] ## 제6조 이메일·메신저  - 외부로 첨부파일을 보낼 때는 자동으로 팀장 결재가 요청됩니 ...
 - [research] ## 제3조 연구윤리  - 다른 사람의 글·데이터를 쓸 때는 반드시 출처를 밝힙니다. -  ...

───[ security만 검색 ]───
 - [security] - 문서 작성자는 작성 시 반드시 보안등급을 표시해야 합니다. - 대외비 문서 출력 시 워 ...
 - [security] ## 제6조 이메일·메신저  - 외부로 첨부파일을 보낼 때는 자동으로 팀장 결재가 요청됩니 ...
 - [security] ## 제4조 저장매체·기기  - 개인 USB 사용은 금지되며, IT팀이 지급한 보안 USB ...


### 셀 15 — 규정에 없는 질문은 점수로 걸러내기

관련도 점수가 기준(`THRESHOLD`)보다 낮은 청크는 버립니다. 남는 게 없으면 "규정에서 확인되지 않습니다"로 안내합니다.

In [21]:
THRESHOLD = 0.3   # TODO: 0.3에서 시작해서 결과를 보며 조정해 보세요

def search_with_threshold(query, k=3):
    pairs = hri_db.similarity_search_with_relevance_scores(query, k=k)
    # TODO: 점수 s가 THRESHOLD 이상인 것만 남기는 조건 (힌트: 비교 연산자 >=)
    kept = [(d, s) for d, s in pairs if s >= THRESHOLD]
    print(f"질문: {query}")
    print(f"   점수: {[round(s, 3) for _, s in pairs]}")
    if not kept:
        print("   → 기준을 넘는 청크 없음: '규정에서 확인되지 않습니다. 담당 부서에 문의해 주세요.'\n")
    else:
        for d, s in kept:
            print(f"   → {s:.3f} [{d.metadata['category']} / {d.metadata['department']}] {d.page_content[:40]!r}")
        print()

search_with_threshold("휴가 남은 거 내년에 써도 돼요?")
search_with_threshold("오늘 점심 메뉴 추천해 줘")

질문: 휴가 남은 거 내년에 써도 돼요?
   점수: [0.48, 0.351, 0.332]
   → 0.480 [leave / 인사팀] '## 제3조 연차휴가\n\n| 근속 기간 | 연차 일수 |\n|--------'
   → 0.351 [leave / 인사팀] '## 제5조 경조휴가 (유급)\n\n| 구분 | 휴가 | 경조금 |\n|---'
   → 0.332 [travel / 재무팀] '## 제2조 국내 출장비\n\n| 항목 | 기준 |\n|------|-----'

질문: 오늘 점심 메뉴 추천해 줘
   점수: [0.281, 0.269, 0.25]
   → 기준을 넘는 청크 없음: '규정에서 확인되지 않습니다. 담당 부서에 문의해 주세요.'



> 💡 두 질문의 점수 차이를 보고 `THRESHOLD`를 정합니다. 정답 숫자는 없고, **우리 질문들로 찍어 보고 정하는 것**이 핵심입니다.

# 5️⃣ 단계 — 미션: 현경봇 검색 정확도 측정

### 셀 16 — 미션 ①: 직원 말투 질문 10개 테스트

각 질문의 **정답 분류**를 정해 두고, 1등으로 찾은 청크의 분류가 맞는지 채점합니다.
챗봇을 만들 때 이런 **테스트 질문 세트**를 먼저 만드는 게 실무 방식입니다.

In [22]:
test_questions = [
    ("집에서 일해도 돼요?", "leave"),
    ("휴가 남은 거 내년에 써도 되나요?", "leave"),
    ("애기 태어나면 며칠 쉴 수 있어요?", "leave"),
    ("법카로 술값 계산해도 돼요?", "travel"),
    ("부산 가면 호텔비 얼마까지 나와요?", "travel"),
    ("챗GPT에 회사 자료 넣어도 되나요?", "security"),
    ("회사의 기밀 정보를 외부에 공유해도 되나요?", "security"),   # TODO: 나만의 질문
    ("직장에서의 인권 침해는 어떻게 신고하나요?", "welfare"),    # TODO
    ("연구 프로젝트의 결과물을 외부에 공개해도 되나요?", "research"),   # TODO
    ("직장에서의 성추행은 어떻게 신고하나요?", "welfare"),       # TODO: 분류도 직접
]

correct = 0
for q, answer in test_questions:
    top = hri_db.similarity_search(q, k=1)[0]
    # TODO: 1등 청크의 분류 꼬리표 꺼내기
    got = top.metadata["category"]
    ok = got == answer
    correct += ok
    print(f"{'✅' if ok else '❌'} {q:<28} 정답={answer:<9} 검색={got}")

print(f"\n정확도: {correct}/{len(test_questions)} ({correct / len(test_questions):.0%})")

❌ 집에서 일해도 돼요?                  정답=leave     검색=welfare
✅ 휴가 남은 거 내년에 써도 되나요?          정답=leave     검색=leave
✅ 애기 태어나면 며칠 쉴 수 있어요?          정답=leave     검색=leave
✅ 법카로 술값 계산해도 돼요?              정답=travel    검색=travel
✅ 부산 가면 호텔비 얼마까지 나와요?          정답=travel    검색=travel
✅ 챗GPT에 회사 자료 넣어도 되나요?         정답=security  검색=security
❌ 회사의 기밀 정보를 외부에 공유해도 되나요?     정답=security  검색=research
❌ 직장에서의 인권 침해는 어떻게 신고하나요?      정답=welfare   검색=security
✅ 연구 프로젝트의 결과물을 외부에 공개해도 되나요?  정답=research  검색=research
✅ 직장에서의 성추행은 어떻게 신고하나요?        정답=welfare   검색=welfare

정확도: 7/10 (70%)


> 💡 틀린 질문이 있다면 이유를 생각해 보세요.
> - 질문이 너무 짧거나 줄임말(법카, 비번)이라서?
> - 비슷한 내용이 FAQ에도 있어서? (FAQ로 나왔다면 사실상 정답일 수도 있습니다)
> - 무료 임베딩 모델의 한국어 실력 한계? (OpenAI 모델로 바꾸면 보통 정확도가 올라갑니다)

> ✍️ 틀린 질문과 내가 생각한 이유: 

### 셀 17 — 미션 ②: (도전) 비슷한 청크만 잔뜩 오는 문제 — MMR

`k=3`으로 찾았는데 같은 조항의 비슷한 청크만 3개 오면 정보가 부족합니다.
**MMR(Maximal Marginal Relevance)** 은 "관련 있으면서 **서로 다른**" 청크를 고릅니다.
4교시에서 쓸 **retriever** 형태로 만들어 봅니다.

In [ ]:
q = "해외 출장 갈 때 챙겨야 할 규정 알려줘"

# TODO: 벡터 DB를 retriever로 바꾸는 메서드
sim_retriever = hri_db.as_retriever(search_kwargs={"k": 3})
# TODO: search_type을 "mmr"로
mmr_retriever = hri_db.as_retriever(search_type="mmr", search_kwargs={"k": 3, "fetch_k": 10})

print("───[ 일반 유사도 ]───")
for d in sim_retriever.invoke(q):
    print(f" - [{d.metadata['category']}]", d.page_content[:50].replace("\n", " "), "...")

print("\n───[ MMR ]───")
for d in mmr_retriever.invoke(q):
    print(f" - [{d.metadata['category']}]", d.page_content[:50].replace("\n", " "), "...")

───[ 일반 유사도 ]───
 - [travel] # 현대경제연구소 출장·경비 규정  ※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합 ...
 - [travel] ## 제4조 해외 출장비  | 항목 | 북미·유럽 | 아시아 | 기타 지역 | |----- ...
 - [travel] - 항공권: 이코노미석 (임원은 비행 8시간 이상 시 비즈니스석) - 항공권은 재무팀 지정 ...

───[ MMR ]───
 - [travel] # 현대경제연구소 출장·경비 규정  ※ 교육용 가상 문서입니다. 실제 기관의 규정과 무관합 ...
 - [travel] ## 제4조 해외 출장비  | 항목 | 북미·유럽 | 아시아 | 기타 지역 | |----- ...
 - [security] - 문서 작성자는 작성 시 반드시 보안등급을 표시해야 합니다. - 대외비 문서 출력 시 워 ...


: 

> 💡 해외 출장 질문은 출장비(travel)뿐 아니라 **노트북 반출(security)** 규정도 관련 있습니다. MMR이 더 다양한 근거를 가져오는지 비교해 보세요.

---
# 🎓 정리

| 단계 | 한 일 | 핵심 |
|------|-------|------|
| 1️⃣ 이론 | 키워드 검색 한계 → 의미 검색 | 가까우면 비슷한 뜻 |
| 2️⃣ 기본 | 문장 → 숫자(임베딩), 코사인 유사도 | `embed_query` |
| 3️⃣ 응용 | 저장·검색·k·filter·점수 | `similarity_search`, `..._with_relevance_scores` |
| 4️⃣ 실습 | 사내규정 6개 의미 검색, 임계값 | 직원 말투로도 조항을 찾음 |
| 5️⃣ 미션 | 테스트 질문 세트로 정확도 측정, MMR | 4교시 retriever 준비 |

**한 문장 요약**: 규정 청크를 **의미 좌표**로 저장하면, 직원이 어떤 말투로 물어도 **뜻이 가까운 조항**을 찾을 수 있다.

**다음 시간(4교시)**: 찾아온 청크를 현경봇에게 근거로 넘겨서 **"(근거: 근무·휴가 규정 제4조 / 문의: 인사팀)"** 이 붙은 답변을 만듭니다.

## 🤔 생각해보기

1. 셀 15에서 기준 점수를 너무 높게 잡으면? 너무 낮게 잡으면? 각각 현경봇이 어떻게 행동할까요?
2. 셀 16에서 '법카', '비번' 같은 사내 줄임말을 잘 찾게 하려면 어떤 방법이 있을까요? (힌트: 문서에 무엇을 추가할 수 있을까?)
3. 보안 질문인데 FAQ 청크가 1등으로 나왔습니다. 이건 틀린 걸까요, 맞은 걸까요? 테스트 채점 기준을 어떻게 정하면 좋을까요?

> ✍️ 메모:
> - 1번:
> - 2번:
> - 3번: